#### Madrid, diciembre de 2024

# PROYECTO FINAL - TÉCNICAS DE RECOGIDA DE DATOS

### Master en Big Data Science - Universidad de Navarra
### Proyecto Final - Técnicas de Recogida de Datos
### Daniel Herrera Torres, Diego de Lemos Burgaña

La API de Spotify es una herramienta poderosa que permite a los desarrolladores acceder a un amplio conjunto de datos relacionados con la música, incluyendo información detallada sobre canciones, álbumes y artistas. Este proyecto utiliza la API de Spotify para ofrecer una funcionalidad que permite generar playlists personalizadas en función de los inputs proporcionados por los usuarios. 

Para ello, se utiliza la API de Spotify para obtener datos relevantes sobre canciones y organizarlos de manera que cumplan con los criterios indicados por el usuario:

Género: El usuario puede seleccionar uno o varios géneros musicales que prefiera para su playlist.
Artistas: La herramienta permite incluir o priorizar canciones de artistas seleccionados por el usuario.
Rango de años: El usuario puede definir un intervalo temporal, asegurando que las canciones seleccionadas pertenecen a un periodo específico.
Duración total: Se puede establecer una duración mínima para el playlist, permitiendo generar listas que se ajusten al tiempo deseado.

En resumen éste código es un script de Python que se conecta a la API de Spotify usando el flujo de credenciales de cliente (Client Credentials Flow), con el propósito de generar un playlist personalizado e interactivo para guardarlo en un DataFrame y, posteriormente, exportarlo a un archivo CSV. Además, muestra las primeras 10 canciones del playlist en el notebook para visualizarlas inmediatamente.

In [1]:
# -----------------------
# Instalar e importar las bibliotecas necesarias para el trabajo
# -----------------------
!pip install spotipy pandas --quiet

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd

# -----------------------
# Configuramos las credenciales generadas para usar la API de Spotify 
# -----------------------
client_id = "938963529f6d45428070eb72ea199389"
client_secret = "3eae28a6a1604c5392ae2cc505311e1b"

# -----------------------
# Conexión y autenticación a la API de Spotify
# -----------------------
client_credentials_manager = SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

# -----------------------
# Definimos la función para transformar el formato de los datos de duración que aparecerán en el DataFrame
# -----------------------
def convertir_duracion(ms):
    minutos = ms // (60 * 1000)
    segundos = (ms % (60 * 1000)) // 1000
    return f"{minutos}:{segundos:02d}"

# -----------------------
# Definimos la función para buscar canciones según los filtros elegidos por el usuario 
# -----------------------
def buscar_canciones_por_filtros(filtro, valor_filtro, release_range, min_duration_minutes):
    canciones = []
    offset = 0  # Para manejar múltiples solicitudes
    total_duration = 0  # Duración total de la playlist en ms
    
    while total_duration < min_duration_minutes * 60 * 1000:  # Mientras no lleguemos a la duración deseada
        if filtro == "genero":
            query = f"genre:{valor_filtro}"
        elif filtro == "artista":
            query = f"artist:{valor_filtro}"
        
        resultados = sp.search(q=query, limit=50, offset=offset)
        if not resultados['tracks']['items']:
            break  # Si no hay más resultados, salimos del bucle
        
        for item in resultados['tracks']['items']:
            release_year = int(item['album']['release_date'][:4])
            if release_range[0] <= release_year <= release_range[1]:
                duration_ms = item['duration_ms']  # Duración en milisegundos
                cancion = {
                    'Nombre': item['name'],
                    'Artista': ', '.join(artist['name'] for artist in item['artists']),
                    'Álbum': item['album']['name'],
                    'Año de Lanzamiento': release_year,
                    'Duración': convertir_duracion(duration_ms),  # Formato "minutos:segundos"
                    'Popularidad': item['popularity'],
                    'URL': item['external_urls']['spotify']
                }
                canciones.append(cancion)
                total_duration += duration_ms
        
        offset += 50  # Incrementar el offset para la próxima solicitud
    
    # Convertir a DataFrame y eliminar duplicados
    df = pd.DataFrame(canciones)
    return df.drop_duplicates(subset='Nombre')

# -----------------------
# Generar los inputs para que el usuario construya su playlist a medida
# -----------------------
print("¡Crea tu playlist personalizada!")

# Input para elegir entre Género o Artista
filtro = input("Elige el filtro de búsqueda: ¿Por 'genero' o por 'artista'? ").strip().lower()

if filtro == "genero":
    valor_filtro = input("Ingresa el género (ej: rock, pop, jazz): ").strip()
elif filtro == "artista":
    valor_filtro = input("Ingresa el nombre del artista: ").strip()
else:
    print("Opción no válida. Eligiendo 'genero' por defecto.")
    filtro = "genero"
    valor_filtro = input("Ingresa el género (ej: rock, pop, jazz): ").strip()

# Input para elegir rango de años de lanzamiento
anio_inicio = int(input("Ingresa el año de lanzamiento inicial para las canciones de tu playlist(ej: 2000): "))
anio_fin = int(input("Ingresa el año de lanzamiento final para las canciones de tu playlist(ej: 2024): "))
duracion_minutos = int(input("Ingresa la duración mínima deseada para tu playlist en minutos (ej: 120): "))

# Buscar canciones con los filtros proporcionados
playlist_df = buscar_canciones_por_filtros(filtro, valor_filtro, (anio_inicio, anio_fin), duracion_minutos)

# -----------------------
# En caso de encontrar datos construir el DataFrame y exportarlo a un archivo CSV
# -----------------------
if not playlist_df.empty:
    # Exportar el DataFrame a un archivo CSV
    nombre_archivo = f"playlist_{filtro}_{valor_filtro}_{anio_inicio}-{anio_fin}_{duracion_minutos}min.csv"
    playlist_df.to_csv(nombre_archivo, index=False)
    print(f"\nPlaylist generada con éxito y exportada a '{nombre_archivo}'.")

    # -----------------------
    # Mostrar las primeras 10 filas del DataFrame en el notebook
    # -----------------------
    print("\nPrimeras 10 canciones de la playlist:")
    display(playlist_df.head(10))  # Usando display() para mostrar en Jupyter Notebook

    # -----------------------
    # Mostrar información de duración total
    # -----------------------
    duracion_total = playlist_df['Duración'].apply(
        lambda x: int(x.split(":")[0]) * 60 + int(x.split(":")[1])
    ).sum() / 60  # Convertir a minutos
    print(f"La playlist tiene una duración total de aproximadamente {duracion_total:.2f} minutos.")
else:
    print("\nNo se encontraron canciones con los filtros seleccionados.")


¡Crea tu playlist personalizada!


Elige el filtro de búsqueda: ¿Por 'genero' o por 'artista'?  genero
Ingresa el género (ej: rock, pop, jazz):  country
Ingresa el año de lanzamiento inicial para las canciones de tu playlist(ej: 2000):  2010
Ingresa el año de lanzamiento final para las canciones de tu playlist(ej: 2024):  2024
Ingresa la duración mínima deseada para tu playlist en minutos (ej: 120):  120



Playlist generada con éxito y exportada a 'playlist_genero_country_2010-2024_120min.csv'.

Primeras 10 canciones de la playlist:


,Nombre,Artista,Álbum,Año de Lanzamiento,Duración,Popularidad,URL
0,Need You Now,Lady A,Need You Now,2010,3:56,72,https://open.spotify.com/track/7GAaTpSoTWUTbP2...
1,Fast Car,Luke Combs,Gettin' Old,2023,4:25,81,https://open.spotify.com/track/1Lo0QY9cvc8sUB2...
2,I Remember Everything (feat. Kacey Musgraves),"Zach Bryan, Kacey Musgraves",Zach Bryan,2023,3:47,85,https://open.spotify.com/track/4KULAymBBJcPRpk...
3,i was all over her,salvia palth,melanchole,2013,2:41,80,https://open.spotify.com/track/6mSnSuOhgHHohqe...
4,Tennessee Whiskey,Chris Stapleton,Traveller,2015,4:53,80,https://open.spotify.com/track/3fqwjXwUGN6vbzI...
5,"Baby, It's Cold Outside (feat. Meghan Trainor)","Brett Eldredge, Meghan Trainor",Glow,2016,2:53,77,https://open.spotify.com/track/5Q2P43CJra0uRAo...
6,Pink Skies,Zach Bryan,Pink Skies,2024,3:14,84,https://open.spotify.com/track/4ZJ4vzLQekI0Wnt...
7,(dream),salvia palth,melanchole,2013,1:24,77,https://open.spotify.com/track/17YNmMy03QLPirm...
8,Something in the Orange,Zach Bryan,Something in the Orange,2022,3:48,84,https://open.spotify.com/track/3WMj8moIAXJhHsy...
9,forwards beckon rebound,Adrianne Lenker,songs,2020,3:09,76,https://open.spotify.com/track/6PBanBy8L1K2Ry4...


La playlist tiene una duración total de aproximadamente 132.98 minutos.
